# 原型与原型链

学习目标：能沿原型链解释属性访问，区分构造函数的 prototype 和实例原型，并合理共享行为。

前置知识：对象属性、属性描述符、this、普通函数和 new 调用。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 文件使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。以下命令均从此目录运行；每个入口使用独立 Node.js 进程。

配套脚本：位于 scripts/13-prototypes/。

1. [lookup.mjs](scripts/13-prototypes/lookup.mjs)：查找、自有属性与空原型。
2. [shadowing.mjs](scripts/13-prototypes/shadowing.mjs)：遮蔽、删除和继承访问器。
3. [readonly-error.mjs](scripts/13-prototypes/readonly-error.mjs)：继承不可写属性的独立反例。
4. [constructor-prototype.mjs](scripts/13-prototypes/constructor-prototype.mjs)：构造与共享方法。
5. [instance-checks.mjs](scripts/13-prototypes/instance-checks.mjs)：未执行构造也可具有同一原型。
6. [sharing-and-composition.mjs](scripts/13-prototypes/sharing-and-composition.mjs)：共享引用边界与组合行为。

## 1 从对象查到原型

读取 note.category 时，自有属性里没有这个键，查找会沿下图向右继续。

对象的原型（prototype）是参与继承查找的另一个对象或 null。读取普通属性先找自有属性，找不到就沿原型链继续；到 null 仍没有匹配属性时得到 undefined。Object.create 明确选择新对象的原型，Object.getPrototypeOf 读取原型。

Object.create 不会执行某个构造函数，也不会把原型属性复制到对象上。Object.hasOwn 只判断自有属性，in 会沿原型链查找。无原型对象适合只需要自身键的字典，但没有继承的 toString 等方法。

![属性读取沿原型链查找。实线箭头表示对象的原型；先检查当前对象的自有属性。](image/illustration/13-01-prototype-lookup.svg)

图示说明：依据普通属性查找规则自绘。这里 base 由普通对象字面量创建，所以其原型是 Object.prototype。

下面用 Object.hasOwn、in 与 missing 分别观察自有属性、整条链上的命中和查找失败。

[lookup.mjs](scripts/13-prototypes/lookup.mjs)：

```javascript
const base = { category: "课程", describe() { return this.title + ":" + this.category; } };
const note = Object.create(base);
note.title = "原型";
console.log(note.describe());
console.log(Object.getPrototypeOf(note) === base);
console.log(Object.hasOwn(note, "category"), "category" in note, note.missing);
console.log(Object.getPrototypeOf(Object.prototype) === null);
const dictionary = Object.create(null);
dictionary.topic = "继承";
console.log(Object.getPrototypeOf(dictionary), typeof dictionary.toString);

// 按本例输入运行，输出依次为：
// 原型:课程
// true
// false true undefined
// true
// null undefined
```

Step 1：运行本节示例。

```bash
node scripts/13-prototypes/lookup.mjs
```

## 2 属性遮蔽与赋值边界

对象自己的同名属性会遮蔽（shadow）原型属性，删除自有属性后又能看到继承属性。普通可写数据属性下，给可扩展实例赋值通常创建或更新实例自身属性，而不是修改原型。

这不是所有赋值的通则：继承的 setter 会以原实例为 this 执行；继承的不可写数据属性会阻止普通赋值，在严格模式下抛 TypeError。读取继承 getter 时 this 也仍是最初的接收者。

[shadowing.mjs](scripts/13-prototypes/shadowing.mjs)：

```javascript
const defaults = { level: 1 };
const student = Object.create(defaults);
student.level = 2;
console.log(student.level, defaults.level, Object.hasOwn(student, "level"));
delete student.level;
console.log(student.level, Object.hasOwn(student, "level"));
const accessors = {
  set score(value) { this.savedScore = value; },
  get score() { return this.savedScore; }
};
const report = Object.create(accessors);
report.score = 8;
console.log(report.score, Object.hasOwn(report, "score"), Object.hasOwn(report, "savedScore"));

// 按本例输入运行，输出依次为：
// 2 1 true
// 1 false
// 8 false true
```

Step 1：运行本节示例。

```bash
node scripts/13-prototypes/shadowing.mjs
```

[readonly-error.mjs](scripts/13-prototypes/readonly-error.mjs)：

```javascript
const base = Object.create(null, { id: { value: 1, writable: false } });
const child = Object.create(base);
child.id = 2;

// 独立运行：退出状态为 1；诊断包含 TypeError；Cannot assign to read only property 'id'。
```

Step 1：单独运行反例，预期非零退出。

```bash
node scripts/13-prototypes/readonly-error.mjs
```

## 3 构造函数、prototype 与 new

同一个单词 prototype 容易混淆两种关系。先区分构造函数的属性，再看实例真正沿哪条链寻找方法。

对本例的普通构造函数，new Notebook 创建的实例以 Notebook.prototype 为原型；Notebook 自身作为函数对象，其原型则是 Function.prototype。下面的规范原图区分对象原型和公开 prototype 属性，避免把两个关系合并。实例不会自动获得一个用于继承的公开 prototype 属性。

对本例的普通构造函数使用 new，可分成创建实例并设置原型、以实例为 this 执行函数体、按返回值规则选择结果。独立状态放在实例上，共享方法放在构造函数的 prototype 对象上。constructor 属性只是普通可修改属性，不能当成不可伪造的类型证据。

![ECMAScript 2025 Figure 1：构造函数 CF 的 prototype 属性指向 CFp，多个独立实例通过隐式原型链接关联 CFp。](image/illustration/13-02-constructor-prototype.svg)

引用原图：ECMAScript 2025 §4.3.1 Figure 1，© 2025 Ecma International，未改绘。本图实线表示公开 prototype 属性，虚线表示隐式原型链接；与第 1 节自绘图的线型不同，应按各自图例阅读。

把 CF 对应到本例的 Notebook，CFp 对应到 Notebook.prototype，cf₁、cf₂ 对应到 first、second。原图还画出更多独立实例；P1、P2、CFP1、q1、q2 是示例属性名，不是本例需要添加的字段。原图没有展开 Function.prototype 对象，下面仍用代码核对构造函数自身的原型。

下面分别核对 getPrototypeOf(first)、getPrototypeOf(Notebook) 和 first.prototype，区分实例原型、函数对象原型与普通属性访问。

[constructor-prototype.mjs](scripts/13-prototypes/constructor-prototype.mjs)：

```javascript
function Notebook(title) { this.title = title; }
Notebook.prototype.describe = function () { return "笔记:" + this.title; };
const first = new Notebook("对象");
const second = new Notebook("类");
console.log(first.describe(), second.describe());
console.log(first.describe === second.describe, Object.hasOwn(first, "describe"));
console.log(Object.getPrototypeOf(first) === Notebook.prototype);
console.log(Object.getPrototypeOf(Notebook) === Function.prototype);
console.log(first.prototype, first.constructor === Notebook);

// 按本例输入运行，输出依次为：
// 笔记:对象 笔记:类
// true false
// true
// true
// undefined true
```

Step 1：运行本节示例。

```bash
node scripts/13-prototypes/constructor-prototype.mjs
```

## 4 instanceof 判断的是链关系

默认的 instanceof 判断右侧构造函数的 prototype 是否出现在左侧对象的原型链中。它并不证明构造函数曾执行，也不做字段校验。右侧可以通过 Symbol.hasInstance 定制行为，该元编程接口在选修专题展开。

重新替换普通构造函数的 prototype 不会改动已有实例的原型，因此旧实例与新实例可能得到不同结果；本例仅用来观察这一边界，不建议在运行中这样组织类型。跨 Realm 的内置构造函数也可能不是同一个对象；识别数组时用 Array.isArray，Realm 在执行模型章节说明。

[instance-checks.mjs](scripts/13-prototypes/instance-checks.mjs)：

```javascript
function Ticket() { this.initialized = true; }
const constructed = new Ticket();
const linkedOnly = Object.create(Ticket.prototype);
console.log(constructed instanceof Ticket, linkedOnly instanceof Ticket);
console.log(constructed.initialized, linkedOnly.initialized);
const oldPrototype = Ticket.prototype;
Ticket.prototype = {};
console.log(constructed instanceof Ticket, new Ticket() instanceof Ticket);
console.log(Object.getPrototypeOf(constructed) === oldPrototype);

// 按本例输入运行，输出依次为：
// true true
// true undefined
// false true
// true
```

Step 1：运行本节示例。

```bash
node scripts/13-prototypes/instance-checks.mjs
```

## 5 共享行为、独立状态与组合

原型上的可变对象同样会被共享。例如原型保存数组时，两个实例访问的是同一数组；对数组 push 不会自动产生实例副本。需要每个实例独立拥有的列表应在构造时创建。

继承可以用 Object.create 建立行为查找链，组合则让对象保存其他对象或函数并委托操作。下例的 tagged 在构造时连接 readerPrototype；createReader 则显式保存一个 formatter。两者都是设计方式，选择取决于要表达共同类型还是可替换能力。不要为本项目扩展 Object.prototype 或 Array.prototype，以免影响无关对象及未来标准方法。

[sharing-and-composition.mjs](scripts/13-prototypes/sharing-and-composition.mjs)：

```javascript
const sharedPrototype = { tags: [] };
const left = Object.create(sharedPrototype);
const right = Object.create(sharedPrototype);
left.tags.push("共享");
console.log(right.tags.join(","), left.tags === right.tags);
function Draft() { this.tags = []; }
const first = new Draft();
const second = new Draft();
first.tags.push("独立");
console.log(first.tags.length, second.tags.length);
// 2. 继承示例沿原型链查找 read，当前对象提供自己的 title。
const readerPrototype = { read() { return this.title; } };
const taggedPrototype = Object.create(readerPrototype);
taggedPrototype.label = function () { return "[" + this.read() + "]"; };
const tagged = Object.create(taggedPrototype);
tagged.title = "原型链";
console.log(tagged.label());
// 3. 组合示例把格式化函数作为能力传入，不增加原型层级。
function createReader(formatter) {
  return { title: "组合", read() { return formatter(this.title); } };
}
console.log(createReader(title => "[" + title + "]").read());

// 按本例输入运行，输出依次为：
// 共享 true
// 1 0
// [原型链]
// [组合]
```

Step 1：运行本节示例。

```bash
node scripts/13-prototypes/sharing-and-composition.mjs
```

## 本章小结

- 原型链控制查找；遮蔽不等于修改原型，访问器与只读属性有额外规则。
- 函数的 prototype 用于实例原型，函数自身的原型是另一个关系。
- 共享方法与共享可变状态要分开；instanceof 只反映相应的原型链关系。

## 练习

1. 创建两层 Object.create 链，先继承 title，再在末端设置同名 title，最后删除它。可核对标准：输出依次为继承值、自有值、继承值。
2. 将共享数组改成每个实例自己的数组。可核对标准：向第一份添加标签后第二份仍为空，两份数组严格不相等。
3. 创建一个具有 Ticket.prototype 原型但没有 initialized 属性的对象。可核对标准：instanceof 为 true，Object.hasOwn 对 initialized 为 false，并解释这不能证明构造曾执行。
4. 在 note 自身添加 category，再删除该自有属性；每次沿原型图标出实际命中位置，并用 note.category 与 Object.hasOwn 检查。

### 提示

1. 末端对象先不建立 title，再赋值，最后 delete。
2. 让构造过程每次执行数组字面量。
3. 在替换 Ticket.prototype 之前创建并核对。
4. 画图时只改变 note 的字段，不修改 base。


### 参考解析

1. `const base = { title: "继承值" }; const middle = Object.create(base); const last = Object.create(middle);`。依次读取 last.title、设置为“自有值”后读取、删除后读取，结果就是继承值、自有值、继承值。
2. 把 `this.tags = []` 放入每次执行的构造体；first.tags !== second.tags 为 true，修改前者不影响后者。
3. `const linked = Object.create(Ticket.prototype)` 不执行 Ticket 的函数体，因此没有自有 initialized；它仍沿原型链关联当前 Ticket.prototype，所以 instanceof 为 true。
4. 设置 `note.category = "自有分类"` 后命中 note，Object.hasOwn 为 true；delete 后命中 base 的“课程”，Object.hasOwn 为 false。


## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TC39 官方 ECMAScript 2025 分页版 | [§4.3.1 Objects 的 Figure 1](https://tc39.es/ecma262/2025/multipage/overview.html#sec-objects)、[原图](https://tc39.es/ecma262/2025/img/figure-1.svg)、[Copyright Notice](https://tc39.es/ecma262/2025/multipage/copyright-and-software-license.html)（原图未改绘，完整版权许可及免责声明见[随图许可](image/illustration/13-02-constructor-prototype.LICENSE.txt)）；[§10.1.8.1 查找](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-ordinaryget)；[§10.1.9.2 赋值和访问器](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-ordinarysetwithowndescriptor)；[§20.1.2.2 Object.create 与相邻 Object 方法](https://tc39.es/ecma262/2025/multipage/fundamental-objects.html#sec-object.create)；[§7.3.21 instanceof 默认算法](https://tc39.es/ecma262/2025/multipage/abstract-operations.html#sec-ordinaryhasinstance)；[§10.2.2 new](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-ecmascript-function-objects-construct-argumentslist-newtarget)。 |
| MDN 用法对照 | [Inheritance、Constructors、Building longer inheritance chains：继承、共享与内置原型边界](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Guide/Inheritance_and_the_prototype_chain)。 |
